# InsPLAD Explainable QA Gold v2 — quality-first final benchmark

최종 benchmark를 숫자에 맞추지 않고 규칙으로 만듭니다.

- Aligned Core 1,378에서 모든 anomaly 보존
- normal cap = 5개 asset의 normal 개수 중앙값(현재 33)
- cap을 넘는 normal은 DINOv2 feature + k-center로 시각적으로 다양한 대표 샘플 선택
- 선택된 후보를 사람이 시각 검수한 뒤 approve/exclude
- approve된 샘플만 `gold_v2_final`로 freeze

최종 freeze 이후 모든 Image-only / Reference / Vision Expert / Agent 실험은 같은 QA를 사용합니다.


In [ ]:
# 0) 설정
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
SEED = 42
DRIVE_DATA_DIR = None
CANDIDATE_FOLDER = "lvlm_insplad_qa/gold_v2_candidate"
FINAL_FOLDER = "lvlm_insplad_qa/gold_v2_final"
DINO_MODEL_ID = "facebook/dinov2-small"
DINO_BATCH_SIZE = 16

ROBOFLOW_WORKSPACE = "andreluizbvs"
ROBOFLOW_PROJECT = "insplad-seg"
ROBOFLOW_VERSION = 7
ROBOFLOW_FORMAT = "coco-segmentation"

# review_gallery.html을 전부 확인한 뒤에만 True로 바꾸세요.
REVIEW_COMPLETE = False
# 제외할 샘플의 core_id만 넣으면 나머지는 approve 처리됩니다.
EXCLUDE_CORE_IDS = []
EXCLUDE_NOTES = {}  # 예: {"core-...": "severe blur"}

EXPECTED_ZIP_SIZES = {
    "unsupervised_anomaly_detection.zip": 1_170_967_719,
    "supervised_fault_classification.zip": 874_753_167,
}


In [ ]:
# 1) Drive mount + 최신 repo clone + dependencies
import base64, csv, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = Path('/content/drive/MyDrive')
CANDIDATE_ROOT = DRIVE_ROOT / CANDIDATE_FOLDER
FINAL_ROOT = DRIVE_ROOT / FINAL_FOLDER
CANDIDATE_ROOT.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path('/content/lvlm_insplad')
WORK_ROOT = Path('/content/insplad_gold_v2_build')
for p in (REPO_DIR, WORK_ROOT):
    if p.exists(): shutil.rmtree(p)
WORK_ROOT.mkdir(parents=True)

def run(cmd, cwd=None, env=None):
    cmd=[str(x) for x in cmd]
    print('+', ' '.join(cmd[:8]) + (' ...' if len(cmd)>8 else ''))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)

try: token = userdata.get('GITHUB_TOKEN')
except Exception: token = None
if not token: raise RuntimeError('Colab Secret GITHUB_TOKEN이 없습니다.')
auth=base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env=os.environ.copy(); git_env.update({'GIT_CONFIG_COUNT':'1','GIT_CONFIG_KEY_0':'http.extraHeader','GIT_CONFIG_VALUE_0':f'AUTHORIZATION: basic {auth}','GIT_TERMINAL_PROMPT':'0'})
repo_url=f'https://github.com/{REPO}.git'
run(['git','clone','--depth','1','--branch',REPO_REF,repo_url,REPO_DIR],env=git_env)
del token, auth, git_env
run([sys.executable,'-m','pip','install','-q','transformers','accelerate','pillow','pandas','numpy'])
print('repo:', subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO_DIR,text=True).strip())


In [ ]:
# 2) source ZIP 확인
def find_source_zip(filename):
    roots=[]
    if DRIVE_DATA_DIR: roots.append(DRIVE_ROOT/DRIVE_DATA_DIR)
    roots.append(DRIVE_ROOT)
    for root in roots:
        if not root.exists(): continue
        direct=root/filename
        if direct.is_file(): return direct
        matches=list(root.rglob(filename))
        if matches: return matches[0]
    raise FileNotFoundError(filename)

UNSUP_ZIP=find_source_zip('unsupervised_anomaly_detection.zip')
SUP_ZIP=find_source_zip('supervised_fault_classification.zip')
for p in (UNSUP_ZIP,SUP_ZIP):
    if p.stat().st_size != EXPECTED_ZIP_SIZES[p.name]: raise RuntimeError(f'ZIP size mismatch: {p}')
    print('PASS',p)


In [ ]:
# 3) verified Aligned Core 1,378 재생성
MAN=WORK_ROOT/'manifests'; BENCH=WORK_ROOT/'benchmark'; MAN.mkdir(parents=True); BENCH.mkdir(parents=True)
unsup_manifest=MAN/'unsup.csv'; sup_manifest=MAN/'sup.csv'; sup_preview=MAN/'sup_preview.json'
run([sys.executable,REPO_DIR/'scripts/inspect_zip_manifest.py',UNSUP_ZIP,'--out-csv',unsup_manifest,'--out-json',MAN/'unsup.json'])
run([sys.executable,REPO_DIR/'scripts/inspect_zip_manifest.py',SUP_ZIP,'--out-csv',sup_manifest,'--out-json',MAN/'sup.json','--text-preview-json',sup_preview])
overlap_summary=WORK_ROOT/'image_overlap_summary.json'
shutil.copy2(REPO_DIR/'reports/image_overlap_summary.json',overlap_summary)
run([sys.executable,REPO_DIR/'scripts/build_benchmark.py','--anomaly-manifest',unsup_manifest,'--fault-manifest',sup_manifest,'--overlap-summary',overlap_summary,'--annotation-previews',sup_preview,'--repo-root',BENCH,'--seed',str(SEED)])
ALIGNED_CSV=BENCH/'reports/aligned_core.csv'
with ALIGNED_CSV.open(encoding='utf-8-sig',newline='') as f: aligned=list(csv.DictReader(f))
assert len(aligned)==1378, len(aligned)
print('Aligned Core:',len(aligned))


In [ ]:
# 4) Q3용 InsPLAD-seg 준비 (없으면 Q1/Q2만 생성)
SEG_ROOT=CANDIDATE_ROOT/'segmentation_source'/f'insplad-seg-v{ROBOFLOW_VERSION}'
coco_files=list(SEG_ROOT.rglob('_annotations.coco.json')) if SEG_ROOT.exists() else []
if not coco_files:
    try: rf_key=userdata.get('ROBOFLOW_API_KEY')
    except Exception: rf_key=None
    if rf_key:
        run([sys.executable,'-m','pip','install','-q','roboflow'])
        from roboflow import Roboflow
        project=Roboflow(api_key=rf_key).workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
        project.version(ROBOFLOW_VERSION).download(ROBOFLOW_FORMAT,location=str(SEG_ROOT),overwrite=False)
        coco_files=list(SEG_ROOT.rglob('_annotations.coco.json'))
SEG_AVAILABLE=bool(coco_files)
print('segmentation available:',SEG_AVAILABLE)


In [ ]:
# 5) 후보 선택: 모든 anomaly + median-capped DINOv2-diverse normal
SELECTION_DIR=CANDIDATE_ROOT/'selection'
run([sys.executable,REPO_DIR/'scripts/select_final_gold_candidates.py','--aligned-core-csv',ALIGNED_CSV,'--supervised-zip',SUP_ZIP,'--out-dir',SELECTION_DIR,'--model-id',DINO_MODEL_ID,'--batch-size',str(DINO_BATCH_SIZE)])
import pandas as pd
display(pd.read_csv(SELECTION_DIR/'selection_summary.csv'))
print((SELECTION_DIR/'selection_metadata.json').read_text(encoding='utf-8'))


In [ ]:
# 6) 후보 QA + review gallery 생성
cmd=[sys.executable,REPO_DIR/'scripts/build_explainable_qa_gold_v2.py','--aligned-core-csv',ALIGNED_CSV,'--selection-csv',SELECTION_DIR/'selected_candidates.csv','--out-dir',CANDIDATE_ROOT,'--seed',str(SEED),'--seg-version',str(ROBOFLOW_VERSION)]
if SEG_AVAILABLE: cmd += ['--seg-root',SEG_ROOT]
run(cmd)
master_images=sorted(CANDIDATE_ROOT.glob('master_images_*.jsonl'))[-1]
master_qa=sorted(CANDIDATE_ROOT.glob('master_qa_*.jsonl'))[-1]
gallery=[sys.executable,REPO_DIR/'scripts/build_qa_review_gallery.py','--supervised-zip',SUP_ZIP,'--master-images',master_images,'--master-qa',master_qa,'--out-dir',CANDIDATE_ROOT]
loc_images=CANDIDATE_ROOT/'localization_images.jsonl'; loc_qa=sorted(CANDIDATE_ROOT.glob('localization_qa_*.jsonl'))
if loc_images.is_file() and loc_qa: gallery += ['--localization-images',loc_images,'--localization-qa',loc_qa[-1]]
run(gallery)
display(pd.read_csv(CANDIDATE_ROOT/'master_counts.csv'))
print('review gallery:',CANDIDATE_ROOT/'review_gallery.html')
print('manual review:',CANDIDATE_ROOT/'manual_review.csv')


## 7) 사람 검수

`review_gallery.html`에서 모든 후보를 확인합니다. 제외 기준은 모델 성능과 무관하게 미리 고정합니다.

- 대상 물체가 심하게 잘리거나 식별 불가능
- 심한 blur / occlusion으로 GT를 육안 확인하기 어려움
- GT label/defect가 시각적으로 매우 모호함
- 거의 동일한 중복 이미지가 과도하게 반복됨

문제가 있는 카드의 `core_id`만 위의 `EXCLUDE_CORE_IDS`에 적습니다. 성능이 어려워 보인다는 이유만으로 제외하면 안 됩니다.


In [ ]:
# 8) 검수 완료 후 manual_review.csv에 approve/exclude 기록
if not REVIEW_COMPLETE:
    raise RuntimeError('review_gallery.html 전체 검수가 끝난 뒤 REVIEW_COMPLETE=True로 바꾸세요.')
review_path=CANDIDATE_ROOT/'manual_review.csv'
df=pd.read_csv(review_path,dtype=str).fillna('')
exclude=set(EXCLUDE_CORE_IDS)
unknown=sorted(exclude-set(df.core_id))
if unknown: raise RuntimeError(f'unknown core IDs: {unknown[:10]}')
df['quality_status']=df.core_id.map(lambda x:'exclude' if x in exclude else 'approve')
df['quality_issue']=df.core_id.map(lambda x:EXCLUDE_NOTES.get(x,'') if x in exclude else '')
df.to_csv(review_path,index=False)
print(df.quality_status.value_counts())


In [ ]:
# 9) approved만 gold_v2_final로 freeze
if FINAL_ROOT.exists():
    shutil.rmtree(FINAL_ROOT)
run([sys.executable,REPO_DIR/'scripts/freeze_gold_v2_review.py','--candidate-dir',CANDIDATE_ROOT,'--out-dir',FINAL_ROOT])
print((FINAL_ROOT/'build_summary.json').read_text(encoding='utf-8'))
display(pd.read_csv(FINAL_ROOT/'master_counts.csv'))
print('FINAL QA:',FINAL_ROOT)


## Freeze 후 실험 순서

1. Image-only baseline
2. Reference ablation: Normal / Positive / Normal+Positive
3. Vision Expert screening: PatchCore / Dinomaly / AnomalyCLIP / ReConPatch 계열 + ROI/heatmap
4. Best Reference + Best Vision Expert
5. Agent

모든 최종 비교는 `gold_v2_final`의 동일 QA/GT에서 진행합니다.
